# Lab: RAG on Rules of a new sport (Student Version)

Objective: build a simple and complete Retrieval-Augmented Generation (RAG) workflow using the Laws of the Game.

We will:
- Load a rules knowledge base
- Build a lightweight index (embeddings)
- Retrieve the most relevant rules for a question
- Produce a short answer from the retrieved context

This version has blanks for you to complete.

## Setup
We work with the files created in the exercise folder:
- rules.json (knowledge base)
- questions_mcq_student.json (questions)

In [1]:
import json
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_DIR = Path('lab_rag_math/data/excercise')
RULES_PATH = BASE_DIR / 'rules.json'
QUESTIONS_PATH = BASE_DIR / 'questions_mcq_student.json'
CACHE_DIR = BASE_DIR / 'models_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)


First, let's ensure the `lab_rag_math/data/excercise` directory exists and download the `rules.json` and `questions_mcq_student.json` files.

In [2]:
import os
import requests

# Ensure the base directory exists
BASE_DIR = Path('lab_rag_math/data/excercise')
os.makedirs(BASE_DIR, exist_ok=True)

# Download rules.json
rules_url = "https://raw.githubusercontent.com/ferrazzipietro/dataset-nlp-math/main/data_rag/excercise/rules.json"
rules_path = BASE_DIR / 'rules.json'
response = requests.get(rules_url)
response.raise_for_status() # Raise an exception for HTTP errors
with open(rules_path, 'wb') as f:
    f.write(response.content)
print(f"Downloaded rules.json to {rules_path}")

# Download questions_mcq_student.json
questions_url = "https://raw.githubusercontent.com/ferrazzipietro/dataset-nlp-math/main/data_rag/excercise/questions_mcq_student.json"
questions_path = BASE_DIR / 'questions_mcq_student.json'
response = requests.get(questions_url)
response.raise_for_status() # Raise an exception for HTTP errors
with open(questions_path, 'wb') as f:
    f.write(response.content)
print(f"Downloaded questions_mcq_student.json to {questions_path}")


Downloaded rules.json to lab_rag_math/data/excercise/rules.json
Downloaded questions_mcq_student.json to lab_rag_math/data/excercise/questions_mcq_student.json


## Build the Index
We embed each rule paragraph with a sentence embedding model.

In [3]:

rules = json.loads(RULES_PATH.read_text(encoding='utf-8'))
questions = json.loads(QUESTIONS_PATH.read_text(encoding='utf-8'))

print('Rules:', len(rules))
print('Questions:', len(questions))

Rules: 49
Questions: 25


In [4]:
# TODO: initialize the embedder
# Hint: use Qwen/Qwen3-Embedding-0.6B
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")  # replace with a real model

# TODO: build the KB chunks with embeddings
KB_CHUNKS = []
for rule_id, text in rules.items():
    # TODO: compute embedding for text
    embedding = embedder.encode([text])[0].tolist()  # replace
    KB_CHUNKS.append({
        "id": rule_id,
        "text": text,
        "embedding": embedding
    })

print('KB chunks:', len(KB_CHUNKS))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

KB chunks: 49


## Retrieval
Implement the exact functions below (see the solution notebook for reference).

In [5]:
# TODO: implement these functions
def embed_query(query):
    # return a single embedding for the query
    return embedder.encode([query])

def cosine_similarity_matrix(query, kb_emb):
    # return a 1 x N similarity matrix
    query = np.array(query).reshape(1, -1)
    kb_emb = np.array(kb_emb)
    query = query / np.clip(np.linalg.norm(query, axis=1, keepdims=True), 1e-12, None)
    kb_emb = kb_emb / np.clip(np.linalg.norm(kb_emb, axis=1, keepdims=True), 1e-12, None)
    return query @ kb_emb.T

def retrieve_top_n(query, kb_emb, n=3):
    # return top-n chunks with text and similarity
    q = embed_query(query)
    document_embeddings = [chunk['embedding'] for chunk in kb_emb]
    sims = cosine_similarity_matrix(q, document_embeddings)[0]
    top_idx = np.argsort(-sims)[:n]
    return [{
        'text': kb_emb[i]['text'],
        'similarity': float(sims[i])}
        for i in top_idx]

sample_q = questions[0]['question']
top = retrieve_top_n(sample_q, KB_CHUNKS, n=3)
print('Question:', sample_q)
for item in top:
    print(f"- similarity={item['similarity']:.3f}")
    print(f"  {item['text'][:160]}...")

Question: Which statement correctly describes the Footsquare pitch boundary?
- similarity=0.763
  The pitch is circular with a clearly marked boundary line. The boundary line is part of the field of play....
- similarity=0.690
  Footsquare is a team sport played on a rounded pitch with raised baskets instead of goals. The game emphasizes continuous play, precise passing, and 360-degree ...
- similarity=0.669
  Footsquare uses a zone rule instead of offside. Attacking players may not receive the ball inside the opponent's keeper zone....


## Generation: Answer from Retrieved Context
Complete the missing model loading and generation pieces.

In [6]:
# TODO: load the generator model
GEMMA_ID = "unsloth/gemma-3-1B-it"
tokenizer_gemma = AutoTokenizer.from_pretrained(GEMMA_ID, cache_dir=CACHE_DIR)  # replace
model_gemma = AutoModelForCausalLM.from_pretrained(GEMMA_ID, cache_dir=CACHE_DIR, torch_dtype=torch.float16, device_map='auto')  # replace

#QUERIES = [{"query": q["question"]} for q in questions]


system_prompt = """You are a helpful assistant that answers questions about a new sport,
answer only with the letter of the correct choice: 'A', 'B', 'C', 'D' or 'E'. Remember to use just one letter in your answer."""

RESULTS = []
for question in questions:
    query = question['question']
    options = question['options']
    print('Query: ', query)
    top_chunks = retrieve_top_n(query, KB_CHUNKS, n=3)
    for retrieved in top_chunks:
        print(f"Similarity={retrieved['similarity']:.3f}, text={retrieved['text'][:180]}")
    context = [r['text'] for r in top_chunks]

    user_message = f"""Here is some useful context: {'\n'.join(context)}\n\n Here is the query: {query} \n\n Here are the options: {options}
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]

    # TODO: build model inputs from messages
    inputs = tokenizer_gemma.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_gemma.device)  # replace

    # TODO: generate and decode the answer
    outputs = model_gemma.generate(**inputs, max_new_tokens=40)
    answer = tokenizer_gemma.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    RESULTS.append({
        'id': question['id'],
        'query': query,
        'options': options,
        'retrieved': top_chunks,
        'answer': answer
    })
    print()

config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Query:  Which statement correctly describes the Footsquare pitch boundary?
Similarity=0.763, text=The pitch is circular with a clearly marked boundary line. The boundary line is part of the field of play.
Similarity=0.690, text=Footsquare is a team sport played on a rounded pitch with raised baskets instead of goals. The game emphasizes continuous play, precise passing, and 360-degree spatial awareness.
Similarity=0.669, text=Footsquare uses a zone rule instead of offside. Attacking players may not receive the ball inside the opponent's keeper zone.

Query:  What is the radius of the central circle used for kickoff?
Similarity=0.822, text=A central circle with a radius of 6 m marks the kickoff area. A surrounding ring (the control ring) extends from 6 m to 12 m.
Similarity=0.725, text=The match begins with a center kick from the central circle. Opponents must remain outside the control ring.
Similarity=0.658, text=Center kick: starts each period and follows every score.

Query:  Which 

We do not have the labels to understand how the model performs. To have an idea about whether it works well, we are going to use an authomatic Judge (LLM-as-a-judge), to ask it:
1) whether the answer is grounded in the retrieved context
2) whether the retrieved context is reliable for the query.

By doing so, you can get an idea about the performances of the model, and you can use this results to do fix your RAG to get better results.

 Be careful, it can be the case that your Judge is wrong!

In [ ]:
def run_judge(prompt):
    # TODO: implement a function that runs the judge model on the prompt and returns the output
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer_gemma.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model_gemma.device)  # replace

    # TODO: generate and decode the answer
    outputs = model_gemma.generate(**inputs, max_new_tokens=10)
    answer = tokenizer_gemma.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return answer



JUDGMENTS = []
for result in RESULTS:
    query = result["query"]
    answer = result["answer"]
    options = result["options"]
    context = "\n".join([r["text"] for r in result["retrieved"]])
    grounding_prompt = f"""You are an impartial judge evaluating an AI assistant.
    Your task is to determine whether the provided answer is supported by the given context.

    Query: {query}
    Options: {options}
    Context: {context}
    Answer Chosen: {answer}

    Is the chosen answer grounded in and supported by the context?
    Reply strictly with a single word: YES or NO.""" # TODO: replace with a prompt to evaluate if the answer is grounded in the context

    reliability_prompt = f"""You are an impartial judge evaluating a retrieval system.
    Your task is to determine if the retrieved context contains sufficient and relevant information to definitively answer the given query.

    Query: {query}
    Context: {context}

    Does the context provide the necessary information to answer the query?
    Reply strictly with a single word: YES or NO.""" # TODO: replace with a prompt to evaluate if the retrieved context is reliable for the query

    grounded = run_judge(grounding_prompt)
    reliable = run_judge(reliability_prompt)
    JUDGMENTS.append({
        "query": query,
        "grounding": grounded,
        "context_reliability": reliable
    })

print('Judged:', len(JUDGMENTS))
print('Sample:', JUDGMENTS[0] if JUDGMENTS else None)

grounding_success = sum(1 for j in JUDGMENTS if "YES" in j["grounding"].upper())
reliability_success = sum(1 for j in JUDGMENTS if "YES" in j["context_reliability"].upper())

total = len(JUDGMENTS)
overall_score = {
    "grounding_accuracy": grounding_success / total if total > 0 else 0.0,
    "reliability_accuracy": reliability_success / total if total > 0 else 0.0
}

print('Overall score:', overall_score)

Judged: 25
Sample: {'query': 'Which statement correctly describes the Footsquare pitch boundary?', 'grounding': 'YES\n', 'context_reliability': 'NO\n'}
Overall score: {'grounding_accuracy': 1.0, 'reliability_accuracy': 0.56}


In [8]:
import pandas as pd
import csv

submission_rows = []
for result in RESULTS:
  submission_rows.append({
      "id": result['id'],
      "prediction": result['answer'][0]
  })
submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv(BASE_DIR/"exercise_predictions.csv", index=False)

In [9]:
submission_df.head()

,id,prediction
0,Q1,E
1,Q2,A
2,Q3,A
3,Q4,D
4,Q5,B


# How to submit

- Generate a `.csv` with two columns, `id` and `prediction`
- Go to the CodaBench portal https://www.codabench.org/competitions/16004/
- Select "Lab 5 - RAG" under the "My Submissions" page
- Submit a .zip file containing both the submission `.csv` file and the `.py` with your code.
- You should achieve an accuracy of at least 0.75
- We will take into account the number of submissions you do in CodaBench. The fewer, the better.